<a href="https://colab.research.google.com/github/Zayaiken21/Youtube-Blue/blob/main/chatterbox_backend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

print("=" * 72)
print("CHATTERBOX TTS — FULL CLEAN CPU REBUILD")
print("=" * 72)

BASE = Path("/tmp/chatterbox_clean")
PROJECT = BASE / "Chatterbox-TTS-Server"
VENV = BASE / "python310"

def run(cmd, cwd=None, check=True):
    cmd = [str(x) for x in cmd]
    print("\n>", " ".join(cmd))
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}\n"
            + " ".join(cmd)
        )
    return result

print("\nSystem Python:")
print(sys.version)

# ---------------------------------------------------------
# 1. START COMPLETELY CLEAN
# ---------------------------------------------------------

print("\n[1/10] Removing old clean-build folder...")

if BASE.exists():
    shutil.rmtree(BASE)

BASE.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 2. INSTALL UV
# ---------------------------------------------------------

print("\n[2/10] Installing uv...")

run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "uv"
])

# ---------------------------------------------------------
# 3. INSTALL PYTHON 3.10
# ---------------------------------------------------------

print("\n[3/10] Installing Python 3.10...")

run([
    sys.executable,
    "-m",
    "uv",
    "python",
    "install",
    "3.10"
])

# ---------------------------------------------------------
# 4. CREATE SEEDED PYTHON 3.10 VENV
# ---------------------------------------------------------

print("\n[4/10] Creating Python 3.10 environment...")

run([
    sys.executable,
    "-m",
    "uv",
    "venv",
    "--python",
    "3.10",
    "--seed",
    str(VENV)
])

PYTHON = VENV / "bin" / "python"

run([PYTHON, "--version"])
run([PYTHON, "-m", "pip", "--version"])

# ---------------------------------------------------------
# 5. CLONE SERVER
# ---------------------------------------------------------

print("\n[5/10] Cloning Chatterbox TTS Server...")

run([
    "git",
    "clone",
    "--depth",
    "1",
    "https://github.com/devnen/Chatterbox-TTS-Server.git",
    str(PROJECT)
])

# ---------------------------------------------------------
# 6. UPGRADE INSTALL TOOLS
# ---------------------------------------------------------

print("\n[6/10] Updating pip tooling...")

run([
    PYTHON,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "pip",
    "setuptools",
    "wheel"
])

# ---------------------------------------------------------
# 7. INSTALL CPU REQUIREMENTS
# ---------------------------------------------------------

print("\n[7/10] Installing CPU requirements...")

REQ = PROJECT / "requirements.txt"

run([
    PYTHON,
    "-m",
    "pip",
    "install",
    "--no-cache-dir",
    "-r",
    str(REQ)
])

# ---------------------------------------------------------
# 8. FORCE THE WORKING CORE VERSIONS
# ---------------------------------------------------------

print("\n[8/10] Pinning working dependency versions...")

run([
    PYTHON,
    "-m",
    "pip",
    "install",
    "--no-cache-dir",
    "--force-reinstall",
    "torch==2.5.1",
    "torchvision==0.20.1",
    "torchaudio==2.5.1",
    "--index-url",
    "https://download.pytorch.org/whl/cpu"
])

run([
    PYTHON,
    "-m",
    "pip",
    "install",
    "--no-cache-dir",
    "--force-reinstall",
    "numpy==1.25.2",
    "safetensors==0.5.3",
    "transformers==4.46.3",
    "diffusers==0.29.0",
    "protobuf==3.20.3",
    "gradio==5.44.1",
    "pykakasi==2.3.0",
    "spacy-pkuseg",
    "pre-commit"
])

# ---------------------------------------------------------
# 9. INSTALL CHATTERBOX + SPECIAL PACKAGES WITHOUT DEPS
# ---------------------------------------------------------

print("\n[9/10] Installing Chatterbox core...")

run([
    PYTHON,
    "-m",
    "pip",
    "install",
    "--no-cache-dir",
    "--no-deps",
    "git+https://github.com/devnen/chatterbox-v2.git@master"
])

run([
    PYTHON,
    "-m",
    "pip",
    "install",
    "--no-cache-dir",
    "--force-reinstall",
    "--no-deps",
    "s3tokenizer==0.3.0"
])

run([
    PYTHON,
    "-m",
    "pip",
    "install",
    "--no-cache-dir",
    "--force-reinstall",
    "--no-deps",
    "onnx==1.16.0"
])

# ---------------------------------------------------------
# 10. VERIFY
# ---------------------------------------------------------

print("\n[10/10] Verifying environment...")

tests = [
    ("PYTHON", "import sys; print(sys.version)"),
    ("TORCH", "import torch; print(torch.__version__)"),
    ("TORCHVISION", "import torchvision; print(torchvision.__version__)"),
    ("TORCHAUDIO", "import torchaudio; print(torchaudio.__version__)"),
    ("NUMPY", "import numpy; print(numpy.__version__)"),
    ("PROTOBUF", "import google.protobuf; print(google.protobuf.__version__)"),
    ("ONNX", "import onnx; print(onnx.__version__)"),
    ("S3TOKENIZER", "import s3tokenizer; print('S3TOKENIZER OK')"),
    ("LLAMA", "from transformers import LlamaModel; print('LLAMA OK')"),
    ("CHATTERBOX", "import chatterbox; print('CHATTERBOX OK')")
]

all_good = True

for name, code in tests:
    print("\nTEST:", name)
    result = run(
        [PYTHON, "-c", code],
        check=False
    )
    if result.returncode != 0:
        all_good = False

print("\n" + "=" * 72)

if all_good:
    print("✅ CHATTERBOX ENVIRONMENT READY")
else:
    print("❌ ONE OR MORE TESTS FAILED")

print("=" * 72)

print("\nPython:")
print(PYTHON)

print("\nProject:")
print(PROJECT)

CHATTERBOX TTS — FULL CLEAN CPU REBUILD

System Python:
3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

[1/10] Removing old clean-build folder...

[2/10] Installing uv...

> /usr/bin/python3 -m pip install --upgrade uv


[3/10] Installing Python 3.10...

> /usr/bin/python3 -m uv python install 3.10
Python 3.10 is already installed


[4/10] Creating Python 3.10 environment...

> /usr/bin/python3 -m uv venv --python 3.10 --seed /tmp/chatterbox_clean/python310
Using CPython 3.10.22
Creating virtual environment with seed packages at: /tmp/chatterbox_clean/python310
 + packaging==26.3
 + pip==26.2.1
 + setuptools==84.0.0
 + wheel==0.48.0
Activate with: source /tmp/chatterbox_clean/python310/bin/activate


> /tmp/chatterbox_clean/python310/bin/python --version
Python 3.10.22


> /tmp/chatterbox_clean/python310/bin/python -m pip --version
pip 26.2.1 from /tmp/chatterbox_clean/python310/lib/python3.10/site-packages/pip (python 3.10)


[5/10] Cloning Chatterbox TTS Server...

> git clone --

In [43]:
import subprocess

PYTHON = "/tmp/chatterbox_clean/python310/bin/python"

commands = [
    [
        PYTHON, "-m", "pip", "install",
        "--force-reinstall",
        "--no-cache-dir",
        "setuptools<81"
    ],
    [
        PYTHON, "-m", "pip", "install",
        "--force-reinstall",
        "--no-cache-dir",
        "resemble-perth==1.0.1"
    ]
]

for cmd in commands:
    print("\nRUNNING:")
    print(" ".join(cmd))

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    print(result.stdout)
    print(result.stderr)
    print("EXIT:", result.returncode)


RUNNING:
/tmp/chatterbox_clean/python310/bin/python -m pip install --force-reinstall --no-cache-dir setuptools<81
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 55.1 MB/s  0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 84.0.0
    Uninstalling setuptools-84.0.0:
      Successfully uninstalled setuptools-84.0.0


EXIT: 0

RUNNING:
/tmp/chatterbox_clean/python310/bin/python -m pip install --force-reinstall --no-cache-dir resemble-perth==1.0.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 153.0 MB/s  0:00:00
  Attempting uninstall: resemble-perth
    Found existing installation: resemble-perth 1.0.1
    Uninstalling resemble-perth-1.0.1:
      Successfully uninstalled resemble-perth-1.0.1


EXIT: 0


In [44]:
import subprocess

PYTHON = "/tmp/chatterbox_clean/python310/bin/python"

code = """
import perth
print("PERTH VERSION:", getattr(perth, "__version__", "unknown"))
print("WATERMARKER:", perth.PerthImplicitWatermarker)
wm = perth.PerthImplicitWatermarker()
print("PERTH WATERMARKER OK")
"""

result = subprocess.run(
    [PYTHON, "-c", code],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("EXIT:", result.returncode)

PERTH VERSION: 1.0.0
WATERMARKER: <class 'perth.perth_net.perth_net_implicit.perth_watermarker.PerthImplicitWatermarker'>
loaded PerthNet (Implicit) at step 250,000
PERTH WATERMARKER OK

/tmp/chatterbox_clean/python310/lib/python3.10/site-packages/perth/perth_net/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/tmp/chatterbox_clean/python310/lib/python3.10/site-packages/perth/perth_net/perth_net_implicit/checkpoint_manager.py:36: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#u

In [48]:
import subprocess
import time
import requests
from pathlib import Path

PYTHON = "/tmp/chatterbox_clean/python310/bin/python"
PROJECT = Path("/tmp/chatterbox_clean/Chatterbox-TTS-Server")

# Stop any old server on 8004
subprocess.run(
    ["pkill", "-f", "server.py"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("Starting Chatterbox in background...")

server_process = subprocess.Popen(
    [PYTHON, "server.py"],
    cwd=str(PROJECT),
    stdout=open("/tmp/chatterbox_server.log", "w"),
    stderr=subprocess.STDOUT,
    text=True
)

print("Server PID:", server_process.pid)

# Wait for port 8004 to respond
for attempt in range(60):
    try:
        r = requests.get("http://127.0.0.1:8004/", timeout=3)
        print("✅ Server is responding:", r.status_code)
        break
    except Exception:
        time.sleep(2)
else:
    print("❌ Server did not become reachable.")
    print("Check logs with:")
    print("!tail -100 /tmp/chatterbox_server.log")

Starting Chatterbox in background...
Server PID: 32393
✅ Server is responding: 200


In [50]:
import requests

r = requests.get("http://127.0.0.1:8004/", timeout=10)

print("Status:", r.status_code)
print("Server reachable:", r.ok)

Status: 200
Server reachable: True


In [ ]:
!npx localtunnel --port 8004

⠙⠹⠸⠼⠴⠦your url is: https://large-poets-train.loca.lt


In [ ]:
# ============================================================
# CHATTERBOX AUTOMATIC STARTUP
# Server + LocalTunnel + GitHub backend-config.json updater
# ============================================================

import os
import re
import json
import time
import base64
import signal
import requests
import subprocess
from pathlib import Path
from datetime import datetime, timezone
from google.colab import userdata

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

PYTHON = "/tmp/chatterbox_clean/python310/bin/python"
PROJECT = Path("/tmp/chatterbox_clean/Chatterbox-TTS-Server")

PORT = 8004
BRANCH = "main"
CONFIG_FILE = "backend-config.json"

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
GITHUB_REPO_URL = userdata.get("GITHUB_REPO_URL")

if not GITHUB_TOKEN:
    raise RuntimeError(
        "Missing Colab secret: GITHUB_TOKEN"
    )

if not GITHUB_REPO_URL:
    raise RuntimeError(
        "Missing Colab secret: GITHUB_REPO_URL"
    )

# ------------------------------------------------------------
# PARSE GITHUB REPO
# ------------------------------------------------------------

repo_match = re.search(
    r"github\.com/([^/]+)/([^/#?]+)",
    GITHUB_REPO_URL
)

if not repo_match:
    raise RuntimeError(
        "GITHUB_REPO_URL is not a valid GitHub repository URL."
    )

GITHUB_OWNER = repo_match.group(1)
GITHUB_REPO = repo_match.group(2)

if GITHUB_REPO.endswith(".git"):
    GITHUB_REPO = GITHUB_REPO[:-4]

print("=" * 72)
print("CHATTERBOX AUTOMATIC STARTUP")
print("=" * 72)

print("Repository:", f"{GITHUB_OWNER}/{GITHUB_REPO}")
print("Port:", PORT)

# ------------------------------------------------------------
# STOP OLD SERVER/TUNNEL PROCESSES
# ------------------------------------------------------------

print("\nStopping old Chatterbox / LocalTunnel processes...")

subprocess.run(
    ["pkill", "-f", "server.py"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

subprocess.run(
    ["pkill", "-f", "localtunnel"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(2)

# ------------------------------------------------------------
# START CHATTERBOX IN BACKGROUND
# ------------------------------------------------------------

print("\nStarting Chatterbox server...")

SERVER_LOG = "/tmp/chatterbox_server.log"

server_log_file = open(
    SERVER_LOG,
    "w"
)

server_process = subprocess.Popen(
    [
        PYTHON,
        "server.py"
    ],
    cwd=str(PROJECT),
    stdout=server_log_file,
    stderr=subprocess.STDOUT,
    text=True
)

print("Server PID:", server_process.pid)

# ------------------------------------------------------------
# WAIT FOR SERVER
# ------------------------------------------------------------

print("\nWaiting for Chatterbox to become available...")

server_ready = False

for attempt in range(90):

    if server_process.poll() is not None:
        print("Server process exited unexpectedly.")
        break

    try:

        r = requests.get(
            f"http://127.0.0.1:{PORT}/api/model-info",
            timeout=5
        )

        if r.status_code == 200:

            print(
                "Server health response:",
                r.status_code
            )

            try:
                info = r.json()
                print(
                    json.dumps(
                        info,
                        indent=2
                    )[:1500]
                )
            except:
                pass

            server_ready = True
            break

    except:
        pass

    if attempt % 10 == 0:
        print(
            f"Still waiting... ({attempt * 2}s)"
        )

    time.sleep(2)

if not server_ready:

    print("\nSERVER FAILED TO BECOME READY")
    print("\nLast server logs:\n")

    try:
        with open(SERVER_LOG, "r") as f:
            logs = f.readlines()
            print("".join(logs[-100:]))
    except:
        pass

    raise RuntimeError(
        "Chatterbox did not become reachable."
    )

print("\n✅ Chatterbox is responding locally.")

# ------------------------------------------------------------
# INSTALL LOCALTUNNEL
# ------------------------------------------------------------

print("\nPreparing LocalTunnel...")

subprocess.run(
    [
        "npm",
        "install",
        "-g",
        "localtunnel"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# ------------------------------------------------------------
# START LOCALTUNNEL
# ------------------------------------------------------------

print("\nStarting LocalTunnel...")

TUNNEL_LOG = "/tmp/localtunnel.log"

tunnel_log_file = open(
    TUNNEL_LOG,
    "w"
)

tunnel_process = subprocess.Popen(
    [
        "lt",
        "--port",
        str(PORT)
    ],
    stdout=tunnel_log_file,
    stderr=subprocess.STDOUT,
    text=True
)

print("Tunnel PID:", tunnel_process.pid)

# ------------------------------------------------------------
# FIND GENERATED LOCALTUNNEL URL
# ------------------------------------------------------------

PUBLIC_URL = None

for attempt in range(60):

    time.sleep(1)

    try:
        text = Path(TUNNEL_LOG).read_text()
    except:
        text = ""

    match = re.search(
        r"https://[a-zA-Z0-9\-]+\.loca\.lt",
        text
    )

    if match:
        PUBLIC_URL = match.group(0)
        break

if not PUBLIC_URL:

    print("\nLocalTunnel output:")
    print(
        Path(TUNNEL_LOG).read_text()
        if Path(TUNNEL_LOG).exists()
        else "No log"
    )

    raise RuntimeError(
        "Could not detect LocalTunnel URL."
    )

print("\n✅ Public tunnel:")
print(PUBLIC_URL)

# ------------------------------------------------------------
# GET LOCALTUNNEL PASSWORD
# ------------------------------------------------------------

TUNNEL_PASSWORD = None

try:

    response = requests.get(
        "https://loca.lt/mytunnelpassword",
        timeout=15
    )

    if response.ok:
        TUNNEL_PASSWORD = response.text.strip()

except Exception as e:
    print(
        "Could not retrieve tunnel password:",
        e
    )

if TUNNEL_PASSWORD:
    print("\nLocalTunnel password/IP:")
    print(TUNNEL_PASSWORD)

# ------------------------------------------------------------
# VERIFY PUBLIC URL
# ------------------------------------------------------------

print("\nTesting public tunnel...")

try:

    public_test = requests.get(
        PUBLIC_URL + "/api/model-info",
        headers={
            "bypass-tunnel-reminder": "true"
        },
        timeout=20
    )

    print(
        "Tunnel HTTP status:",
        public_test.status_code
    )

except Exception as e:

    print(
        "Tunnel test warning:",
        e
    )

# ------------------------------------------------------------
# BUILD backend-config.json
# ------------------------------------------------------------

config = {
    "backend_url": PUBLIC_URL,
    "api_docs": PUBLIC_URL + "/docs",
    "model_info": PUBLIC_URL + "/api/model-info",
    "updated_at": datetime.now(
        timezone.utc
    ).isoformat(),
    "port": PORT
}

print("\nNew backend config:")
print(
    json.dumps(
        config,
        indent=2
    )
)

# ------------------------------------------------------------
# UPDATE GITHUB
# ------------------------------------------------------------

api_url = (
    f"https://api.github.com/repos/"
    f"{GITHUB_OWNER}/"
    f"{GITHUB_REPO}/contents/"
    f"{CONFIG_FILE}"
)

headers = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28"
}

existing = requests.get(
    api_url,
    headers=headers,
    params={
        "ref": BRANCH
    },
    timeout=20
)

sha = None

if existing.status_code == 200:
    sha = existing.json().get("sha")

elif existing.status_code != 404:

    print(existing.text)

    raise RuntimeError(
        "Could not inspect backend-config.json."
    )

encoded_content = base64.b64encode(
    json.dumps(
        config,
        indent=2
    ).encode("utf-8")
).decode("utf-8")

payload = {
    "message":
        "Auto-update Chatterbox backend URL",
    "content":
        encoded_content,
    "branch":
        BRANCH
}

if sha:
    payload["sha"] = sha

update = requests.put(
    api_url,
    headers=headers,
    json=payload,
    timeout=30
)

if update.status_code not in (200, 201):

    print(update.text)

    raise RuntimeError(
        "GitHub backend configuration update failed."
    )

print("\n✅ GitHub backend-config.json updated.")

# ------------------------------------------------------------
# FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("CHATTERBOX IS ONLINE")
print("=" * 72)

print("\nPublic website/server:")
print(PUBLIC_URL)

print("\nAPI docs:")
print(PUBLIC_URL + "/docs")

if TUNNEL_PASSWORD:

    print("\nLocalTunnel access password/IP:")
    print(TUNNEL_PASSWORD)

print("\nGitHub config:")
print(
    f"https://github.com/"
    f"{GITHUB_OWNER}/"
    f"{GITHUB_REPO}/blob/"
    f"{BRANCH}/"
    f"{CONFIG_FILE}"
)

print("""
IMPORTANT:

The server and tunnel are now running in background processes.

Do NOT rerun this cell unless you want to restart them and generate
a new tunnel address.

If LocalTunnel shows its consent screen, enter the password/IP printed
above once in that browser.
""")